# §13.10.4 — 특정 뉴런을 최대로 활성화하는 입력 찾기

> 딥러닝 교재 · 3부 13장 10절 4항 (🐍)
> 선행: §13.10.1(키·값 대응) · §13.10.3(간섭 계산) · §13.10.5(중첩 경고)

## 이 노트북이 답하는 질문

1. **뉴런의 키는 실제로 무엇에 반응하는가?** 입력 공간을 전수 탐색해 반응 지도를 그린다.
2. **뉴런의 값은 무엇을 인출하는가?** 출력 가중치가 미는 토큰과 데이터의 참 전이를 대조한다.
3. **깔끔한 키–값 뉴런만 있는가?** 단의미와 다의미가 한 모델에 공존함을 확인한다.

**예상 실행 시간** CPU 약 90초 (`FAST = True`이면 약 50초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 모델 — 마르코프 언어를 배운 작은 트랜스포머

데이터는 §13.5.4의 마르코프 연쇄(상태 20, 상태당 후속 4개)다. 이 언어의 지식은
"상태 $b$ 다음에는 {네 후속}이 온다"는 전이 사실들이고, §13.10.1의 독법이 옳다면
그 사실들이 MLP의 (키, 값) 쌍으로 저장되어 있어야 한다 — 키는 현재 상태의 검출,
값은 후속 분포의 인출. 어휘가 20개뿐이므로 "최대 활성화 입력 찾기"를 기울기
상승 대신 **전수 탐색**으로 할 수 있다: 모든 (이전, 현재) 바이그램을 넣어 보면 된다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
        np.fill_diagonal(nmask, 0.0)      # 자기 자신은 항상 보인다 (캐시 축출 의미론)
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
V = 20
T_SEQ = 32
rn0 = np.random.default_rng(SEED % 100000)
TRANS_NEXT = np.array([rn0.permutation(V)[:4] for _ in range(V)])
TRANS_P = np.array([0.5, 0.25, 0.15, 0.1])

def markov_batch(B, rn):
    idx = np.zeros((B, T_SEQ), int)
    idx[:, 0] = rn.integers(0, V, B)
    for t in range(1, T_SEQ):
        choice = rn.choice(4, size=B, p=TRANS_P)
        idx[:, t] = TRANS_NEXT[idx[:, t - 1], choice]
    tgt = np.full((B, T_SEQ), -1)
    tgt[:, :-1] = idx[:, 1:]
    return idx, tgt

p, cfg = make_config(V=V, d=48, L=2, h=4, T_max=T_SEQ, pe='learned', seed=11)
rn = np.random.default_rng(500)
STEPS = 250 if FAST else 500
train_lm(p, cfg, lambda: markov_batch(32, rn), steps=STEPS, lr=3e-3)
idx_ev, tgt_ev = markov_batch(256, np.random.default_rng(77))
out = forward(p, cfg, idx_ev, targets=tgt_ev)
print(f"평가 손실 {out['loss']:.3f} (전이 엔트로피 {-(TRANS_P*np.log(TRANS_P)).sum():.3f})")

---
## 2. 키 쪽 — 전수 탐색으로 그리는 반응 지도

모든 바이그램 $(a, b)$를 최소 문맥 $[a, b]$로 넣고, 0층 MLP의 각 뉴런이 위치 $b$에서
내는 활성을 기록한다. $20\times20=400$개 입력의 완전한 반응 지도다.

In [ ]:
pairs = np.array([[a, b] for a in range(V) for b in range(V)])   # 길이 2의 최소 문맥
out_g = forward(p, cfg, pairs, targets=None)
r0 = out_g['cache']['l0_']['mlp'][2][:, 1, :]        # (400, d_ff) — 위치 b의 활성
DFF = r0.shape[1]
maps = r0.T.reshape(DFF, V, V)                        # (뉴런, 이전 a, 현재 b)

# 뉴런 분류 지표: 현재-토큰 열 집중도(단의미 후보)와 반응 영역의 산포(다의미 후보)
col_mass = maps.sum(axis=1)                           # (뉴런, b)
col_share = col_mass.max(1) / (col_mass.sum(1) + 1e-9)
active = maps.reshape(DFF, -1).max(1) > 0.5
n_mono = int(np.argmax(np.where(active, col_share, 0)))          # 가장 집중된 뉴런
poly_pool = np.where(active & (col_share < 0.35))[0]
n_poly = poly_pool[np.argmax(maps[poly_pool].max((1, 2)))]        # 가장 분산된(다의미) 뉴런
b_star = int(col_mass[n_mono].argmax())
print(f"활성 뉴런 {active.sum()}/{DFF} | 열 집중도 분포: 최고 {col_share[active].max():.2f}, "
      f"중앙값 {np.median(col_share[active]):.2f}")
print(f"가장 집중된 뉴런 #{n_mono}: 현재 토큰 {b_star} 검출기 (집중도 {col_share[n_mono]:.2f} — 완전한 단의미는 없다)")
print(f"분산형 뉴런 #{n_poly}: 집중도 {col_share[n_poly]:.2f} — 서로 무관한 여러 패턴에 반응")

---
## 3. 값 쪽 — 이 뉴런이 켜지면 무엇이 인출되는가

뉴런 $i$의 값 벡터는 $W_2$의 $i$번째 행이다(식 13.10.1). 그것이 판독을 지나 어휘에
미치는 영향 $w^{\rm out}_i W_{\rm out}$을 계산해, 데이터의 참 전이(상태 $b^*$의 네
후속)와 대조한다.

In [ ]:
v_dir = p['l0_w2'][n_mono]                            # (d,)
logit_push = v_dir @ p['out']                          # (V,) — 이 뉴런이 미는 토큰들
succ = TRANS_NEXT[b_star]
top4 = np.argsort(logit_push)[::-1][:4]
print(f"상태 {b_star}의 참 후속: {sorted(succ.tolist())} (확률 {TRANS_P})")
print(f"뉴런 #{n_mono}이 미는 상위 4 토큰: {top4.tolist()}")
print(f"→ 일치 {len(set(top4.tolist()) & set(succ.tolist()))}/4")

---
## 4. 교재 그림 — fig_13_10_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 말뭉치 활성 분포
ax = axes[0]
out_full = forward(p, cfg, idx_ev[:128], targets=None)
r_corpus = out_full['cache']['l0_']['mlp'][2]         # (B,T,dff)
for n_id, col, lb in [(n_mono, CB[5], lab(f'집중형 #{n_mono}', f'selective #{n_mono}')),
                      (n_poly, CB[4], lab(f'분산형 #{n_poly}', f'distributed #{n_poly}'))]:
    acts = r_corpus[:, 4:, n_id].ravel()
    ax.hist(acts, bins=40, alpha=0.65, color=col, label=lb, density=True)
ax.set_yscale('log')
ax.set_xlabel(lab('활성값', 'activation'))
ax.set_ylabel(lab('밀도 (로그)', 'density'))
ax.set_title(lab('(a) 활성은 희소하다 — 대부분 0, 가끔 크게', '(a) activation distribution'), fontsize=10)
ax.legend(fontsize=8)

# (b) 단의미 뉴런의 반응 지도
ax = axes[1]
imv = ax.imshow(maps[n_mono], cmap='viridis', aspect='auto')
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.grid(False)
ax.set_xlabel(lab('현재 토큰 $b$', 'current token'))
ax.set_ylabel(lab('이전 토큰 $a$', 'previous token'))
ax.set_title(lab(f'(b) 가장 집중된 뉴런 #{n_mono} — "{b_star}번 상태" 검출기', '(b) most selective'), fontsize=10)

# (c) 값 쪽 대조
ax = axes[2]
cols = [CB[3] if t in succ else CB[7] for t in range(V)]
ax.bar(range(V), logit_push, color=cols)
ax.set_xlabel(lab('어휘 토큰', 'token'))
ax.set_ylabel(lab('뉴런의 로짓 기여 $w^{\\rm out}_i W_{\\rm out}$', 'logit push'))
ax.set_title(lab(f'(c) 값의 정체 — 미는 토큰 = 상태 {b_star}의 참 후속(초록)', '(c) value check'), fontsize=10)

# (d) 다의미 뉴런의 반응 지도
ax = axes[3]
imv = ax.imshow(maps[n_poly], cmap='viridis', aspect='auto')
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.grid(False)
ax.set_xlabel(lab('현재 토큰 $b$', 'current token'))
ax.set_ylabel(lab('이전 토큰 $a$', 'previous token'))
ax.set_title(lab(f'(d) 분산형 뉴런 #{n_poly} — 서로 무관한 여러 반응', '(d) distributed'), fontsize=10)

save_book_fig(fig, 'fig_13_10_4')
plt.show()

> ### 읽는 법
>
> (a) MLP 활성은 희소하다 — 대부분의 입력에서 0(ReLU 문턱 아래), 제 패턴이 오면
> 크게 켜진다. §13.10.2의 문턱-여과가 작동 중이라는 뜻이다.
> (b)–(c) 키–값 독법의 실물 — 그리고 그 근사의 정도. 가장 집중된 뉴런의 키는 "현재
> 토큰 $=b^*$"의 세로줄에 반응하지만 집중도는 0.6에 그치고, 값이 미는 상위 네 토큰은
> 상태 $b^*$의 참 후속 넷 중 **셋**과 일치한다. "전이 사실 하나 ≈ 뉴런 하나"라는
> 그림(§13.10.1, Geva et al. 2021)이 대체로 맞되 정확히는 맞지 않는 것 — 이 근사의
> 잔여가 (d)로 이어진다.
> (d) 같은 모델 안에 반응이 여러 영역에 흩어진 뉴런들이 다수 공존한다(집중도 중앙값
> 0.3대). 사실들은 뉴런 기저가 아니라 **방향들**에 저장되고 있는 것이다 — §13.10.5
> (중첩)의 예고편이며, 해석 작업의 실제 지형이다.

---
## 5. 자기 점검

1. (b)의 검출기가 "이전 토큰"에는 둔감한 이유를 생각해 보라. 마르코프 연쇄에서 다음 토큰 예측에 필요한 정보는 무엇뿐인가?
2. (c)의 로짓 기여 크기 순서가 전이 확률 [0.5, 0.25, 0.15, 0.1]의 순서와 일치하는가? 확인해 보라.
3. 단의미 뉴런을 0으로 고정하고 그 상태에서의 손실 변화를 재 보라. 다른 상태의 손실도 변하는가? (§13.10.6 지식 편집의 국재성 질문이다.)
4. 어휘를 200으로 늘리고 $d_{\rm ff}$를 그대로 두면 (b)류와 (d)류의 비율이 어떻게 변하겠는가? §13.10.5의 중첩 논증으로 예측하고 실험하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `V` | 1절 | 20 | 저장할 사실 수 대 뉴런 수 |
| 검사 층 | 2절 | 0층 MLP | 층별 분업 |
| `STEPS` | 1절 | 500 | 덜 익은 기억의 모습 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")